# 00 · Bike-Sharing — Data Acquisition & Cleaning

## Part 0 — Acquire & Clean (Bike-Sharing hourly demand)

UCI Bike-Sharing: **17,379 hourly records** of bike rentals in Washington D.C., 2011–2012, with
weather and calendar context. Target is **`cnt`** (rentals that hour). Unlike the finance series,
this dataset has **exogenous drivers** (weather, hour, season) — and three things to fix first:

| trap | what | fix |
|---|---|---|
| **Target leakage** | `cnt = casual + registered` *exactly* | drop `casual`/`registered` from predictors |
| **Integer-coded categoricals** | `season`,`weathersit`,`hr`,… are ints but categorical | recast to `category` |
| **Normalized weather** | `temp`/`hum`/`wind` scaled to [0,1] | restore real units (°C, %, km/h) |

In [1]:
import sys, pathlib, warnings
warnings.filterwarnings("ignore", category=FutureWarning)
ROOT = pathlib.Path.cwd(); ROOT = ROOT if (ROOT / "src").exists() else ROOT.parent
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src import data, eda
eda.set_style()
pd.set_option("display.width", 120, "display.max_columns", 30)
print("setup ok | numpy", np.__version__, "| pandas", pd.__version__)


setup ok | numpy 2.1.3 | pandas 2.3.3


### 1. The leakage trap — `cnt = casual + registered`

In [2]:
raw = data.load_raw()
print("shape:", raw.shape, "| missing values:", int(raw.isna().sum().sum()))
print("cnt == casual + registered for ALL rows?", bool((raw.cnt == raw.casual + raw.registered).all()))
print("-> casual & registered are the target split; using them as features LEAKS the answer.")

shape: (17379, 17) | missing values: 0
cnt == casual + registered for ALL rows? True
-> casual & registered are the target split; using them as features LEAKS the answer.


### 2. Integer-coded categoricals
Stored as numbers, but a `weathersit` of 3 isn't '3 units' of anything — it's a category.

In [3]:
print("season   :", raw.season.value_counts().sort_index().to_dict(), "(1=spring..4=winter)")
print("weathersit:", raw.weathersit.value_counts().sort_index().to_dict(), "(1=clear..4=heavy rain)")
print("\nweathersit=4 (heavy rain) has only", int((raw.weathersit==4).sum()), "rows -> a rare category you can't reliably model.")

season   : {1: 4242, 2: 4409, 3: 4496, 4: 4232} (1=spring..4=winter)
weathersit: {1: 11413, 2: 4544, 3: 1419, 4: 3} (1=clear..4=heavy rain)

weathersit=4 (heavy rain) has only 3 rows -> a rare category you can't reliably model.


### 3. Clean it
`data.clean()` builds a datetime index from `dteday + hr`, de-normalizes the weather to real units,
adds readable category labels, recasts the coded categoricals, and drops the row id.

In [4]:
df = data.clean()
print("clean shape:", df.shape, "| index:", df.index.min(), "->", df.index.max())
print("real-unit weather (first row): %.1f°C, feels %.1f°C, %.0f%% hum, %.1f km/h"
      % (df.temp_C[0], df.atemp_C[0], df.hum_pct[0], df.wind_kmh[0]))
df[["season_name","weather_name","year","temp_C","hum_pct","cnt"]].head(3)

clean shape: (17379, 22) | index: 2011-01-01 00:00:00 -> 2012-12-31 23:00:00
real-unit weather (first row): 9.8°C, feels 14.4°C, 81% hum, 0.0 km/h


,season_name,weather_name,year,temp_C,hum_pct,cnt
datetime,,,,,,
2011-01-01 00:00:00,spring,clear,2011,9.84,81.0,16
2011-01-01 01:00:00,spring,clear,2011,9.02,80.0,40
2011-01-01 02:00:00,spring,clear,2011,9.02,80.0,32


### 4. Index hygiene — the hourly grid has gaps
A 2-year hourly grid should have ~17,520 slots, but we have 17,379 — some hours are simply absent.
That matters for the time-series work later (Part 2), where lags assume a regular grid.

In [5]:
print("missing hourly slots:", data.missing_hours(df))
X, y = data.features_target(df)
print("model-ready features (leakage + target removed):", X.shape[1], "columns")

missing hourly slots: 165
model-ready features (leakage + target removed): 19 columns


### 5. Persist

In [6]:
data.build_processed()
print("wrote data/processed/bike_clean.csv  — Part 0 complete.")

wrote data/processed/bike_clean.csv  — Part 0 complete.